In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/d/giladnavok/deep-learning-course-project/')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import *
from src.models.architectures.AdditiveHybridGaborResNet18 import *
from src.utils.training import *
from src.utils.visualization import *

from typing import Dict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.6/87.6 kB 2.8 MB/s eta 0:00:00


Note: you may need to restart the kernel to use updated packages.


====================== Hyperparameters =======================
N_EPOCHS: 200
T_MAX: 200
CRITERION: CrossEntropyLoss()
DEVICE: cuda
SEED: 42
BATCH_SIZE: 128
LR: 0.001
MOMENTUM: 0.9
WEIGHT_DECAY: 0.0001
Setting seed to 42


In [2]:
DEBUG = False
SKIP_TRAINING = False
EXP_NAME = "maxhybridgabor_acc_layers_vs_n_samples"
print(f"Starting experiment {EXP_NAME}. DEBUG={DEBUG}, SKIP_TRAINING={SKIP_TRAINING}")

device = DEVICE

layers = [2, 3, 4, 5]
n_samples_per_class_train = [10, 50, 100, 200, 300, 400, 500, 1000]

Starting experiment maxhybridgabor_acc_layers_vs_n_samples. DEBUG=False, SKIP_TRAINING=False


In [3]:


# Train and save model for each data size
test_accs_L : dict[int, float] = {}
for L in layers:
    test_accs : dict[int, float] = {}
    for n_samples in n_samples_per_class_train:
        # Initialize model
        model = MakeAdditiveHybridGaborResNet18(L=L).to(device)
        MODEL_NAME = f"AdditiveHybridGaborResNet18_L{L}_{n_samples}"
    
        total_params, model_size_mb = get_model_summary(model)
        print(f"Total Parameters: {total_params:,}")
        print(f"Model Size: {model_size_mb:.2f} MB")
    
        # Get data loaders
        trainloader, valloader, testloader, train_set, val_set, test_set = get_cifar10_loaders_and_splits(
        n_samples_per_class_train=n_samples
        )
        optimizer, scheduler = get_optimizer_and_scheduler(model, optimizer_type='sgd-hybrid-gabor')
    
        # Train model on n_samples
        if not SKIP_TRAINING:
            train_model(
                model=model,
                trainloader=trainloader,
                valloader=valloader,
                optimizer=optimizer,
                scheduler=scheduler,
                device=device,
                experiment_name=EXP_NAME,
                model_name=MODEL_NAME,
                val_accuracy_storing_threshold=0,
                print_progress_every=50,
                DEBUG=DEBUG
            )
    
            # Save accuracy on test set
            debug_suff = "_DEBUG" if DEBUG else ""
            load_weights(model, experiment_name=EXP_NAME, model_name=(MODEL_NAME+ debug_suff), device=device)
            test_acc = calculate_accuracy(model, testloader, device)
            test_accs[n_samples] = test_acc
            print(f'Final test accuracy is: {test_acc:.3f}')
            
        test_accs_L[L] = test_accs

Total Parameters: 11,200,080
Model Size: 42.72 MB


  0%|          | 0.00/170M [00:00<?, ?B/s]

  0%|          | 32.8k/170M [00:00<14:59, 190kB/s]

  0%|          | 65.5k/170M [00:00<14:50, 191kB/s]

  0%|          | 98.3k/170M [00:00<14:51, 191kB/s]

...

Using default 500 samples per class for val.


Original train-val size: 50000
Train size: 1000
Val size: 5000
Test size: 10000
Samples per class (train): Counter({np.int64(7): 100, np.int64(1): 100, np.int64(8): 100, np.int64(4): 100, np.int64(3): 100, np.int64(9): 100, np.int64(5): 100, np.int64(0): 100, np.int64(2): 100, np.int64(6): 100})
Samples per class (val): Counter({np.int64(1): 500, np.int64(9): 500, np.int64(2): 500, np.int64(4): 500, np.int64(6): 500, np.int64(5): 500, np.int64(0): 500, np.int64(3): 500, np.int64(8): 500, np.int64(7): 500})
Model weights will be saved to: /kaggle/working/artifacts/checkpoints/maxhybridgabor_acc_layers_vs_n_samples_AdditiveHybridGaborResNet18_L4_100.pth
Stats will be saved to: /kaggle/working/artifacts/stats/maxhybridgabor_acc_layers_vs_n_samples_AdditiveHybridGaborResNet18_L4_100.pkl

=== Starting Training: AdditiveHybridGaborResNet18_L4_100 with 200 epochs ===


    --> New Best Saved: 10.00%
Epoch 1/200 | Loss: nan | Val Acc: 10.00%


Epoch 51/200 | Loss: nan | Val Acc: 10.00%


Epoch 101/200 | Loss: nan | Val Acc: 10.00%


In [ ]:
# Plot accuracy vs data size
plt.figure(figsize=(10, 5))
for L in layers:
    plt.plot(list(test_accs_L[L].keys()), list(test_accs_L[L].values()), marker='o', linestyle='-')
    
plt.xlabel('Number of Training Samples per Class')
plt.ylabel('Test Accuracy')
plt.xscale('log')
plt.title('Accuracy vs Number of Training Samples per Class')
plt.grid(True)
if not SKIP_TRAINING:
    plt.savefig(FIGURES_PATH / 'maxhybridgabor_acc_layers_vs_n_samples.png')
plt.show()